# Robustness of Dictionary-Based Sentiment Scoring<br>for Monetary Policy Shock Identification

**Replication & Extension of Aruoba & Drechsel (2024, *Econometrica*)**

---

## Motivation

Aruoba & Drechsel (2024) identify monetary policy shocks by:

1. Scoring sentiment for 296 economic concepts using the **Loughran-McDonald dictionary** within **±10-word windows** around each concept mention
2. Computing the first principal component (PC1) of these 296 sentiment indicators
3. Ridge-regressing FFR target changes on Greenbook numerical forecasts + sentiment PC1
4. Using the residuals as monetary policy shocks

Their main specification achieves a deviance ratio of **0.94** — meaning 94% of FFR variation is attributed to systematic policy.

### What This Notebook Tests

The dictionary method implicitly assumes that **20 words of local context** (±10 words) are sufficient to determine the directional tone of any concept reference. But what if the relevant context spans paragraphs? What if the dictionary misses nuance that a human reader would catch?

**We test this assumption** by having an LLM (DeepSeek V4 Pro) read the **entire Greenbook1 document** (the staff's qualitative assessment, 8K–24K words) and score the **same 296 concepts** with full document context. This is NOT "LLM replacing dictionary" — it is a robustness check on whether the narrow window is a binding constraint.

**Four checks, one API-dependent:**

| Check | Question | API Cost |
|-------|----------|----------|
| **A** | Does full-context LLM scoring produce a PC1 comparable to window-based dictionary scoring? | ~211 calls |
| **B** | Is the first-stage fit stable across Fed chair regimes? | 0 |
| **C** | Is the signal driven by concept identities or just overall text tone? | 0 |
| **D** | How correlated are the shock series from both methods? | 0 |

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats as scipy_stats
import warnings, json
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
print(f"Project root: {PROJECT_ROOT}")

## 0. Data Loading

Load all four data sources and merge on meeting date.

In [ ]:
# --- 1. FFR target changes ---
ffr_path = PROJECT_ROOT / "data" / "create_sentiments" / "Input_files" / "FFR_Target.csv"
df_ffr = pd.read_csv(ffr_path)

def excel_to_date(serial):
    return (datetime(1899, 12, 30) + timedelta(days=int(serial))).strftime("%Y-%m-%d")

df_ffr["meeting_date"] = df_ffr["Meeting_Date"].apply(excel_to_date)
print(f"FFR data: {len(df_ffr)} meetings, {df_ffr['meeting_date'].min()} to {df_ffr['meeting_date'].max()}")
print(f"  FFR_change: mean={df_ffr['FFR_change'].mean():.3f}, std={df_ffr['FFR_change'].std():.3f}")

# --- 2. Greenbook numerical forecasts ---
reg_path = PROJECT_ROOT / "data" / "create_sentiments" / "Input_files" / "Greenbook_Regressioninput_Original.csv"
df_reg = pd.read_csv(reg_path)
df_reg["meeting_date"] = df_reg["meeting_date"].str.replace("_", "-")
meta_cols = ["Unnamed: 0", "meeting_date", "greenbook_date", "year_gb", "quarter_gb"]
forecast_cols = [c for c in df_reg.columns if c not in meta_cols]
print(f"Greenbook forecasts: {len(df_reg)} meetings, {len(forecast_cols)} forecast variables")

# --- 3. Dictionary sentiment (original, ±10 word window) ---
dict_sent_path = PROJECT_ROOT / "data" / "replication_files" / "Sentiment_Final" / "total_sentiment_file_std10.csv"
df_dict_sent = pd.read_csv(dict_sent_path)
id_col = df_dict_sent.columns[0]
df_dict_sent["meeting_date"] = df_dict_sent[id_col].astype(str).str.replace("_", "-")
dict_concept_cols = [c for c in df_dict_sent.columns if c not in (id_col, "meeting_date")]
print(f"Dictionary sentiment: {len(df_dict_sent)} meetings, {len(dict_concept_cols)} concepts")

# --- 4. LLM sentiment (new, same concept list) ---
llm_sent_path = PROJECT_ROOT / "data" / "processed" / "sentiment_llm_v2.csv"
if llm_sent_path.exists():
    df_llm_sent = pd.read_csv(llm_sent_path)
    df_llm_sent["meeting_date"] = df_llm_sent["meeting_date"].astype(str)
    llm_concept_cols = [c for c in df_llm_sent.columns if c != "meeting_date"]
    print(f"LLM sentiment: {len(df_llm_sent)} meetings, {len(llm_concept_cols)} concepts")
    llm_available = True
else:
    print(f"WARNING: {llm_sent_path} not found. Run Step 1a first.")
    llm_available = False
    df_llm_sent = None
    llm_concept_cols = []

In [ ]:
# --- 5. Merge all data ---
df = df_ffr[["meeting_date", "FFR_change"]].copy()
df = df.merge(df_reg[["meeting_date"] + forecast_cols], on="meeting_date", how="inner")
df = df.merge(df_dict_sent[["meeting_date"] + dict_concept_cols], on="meeting_date", how="inner")
print(f"After FFR + forecasts + dict_sent: {len(df)} meetings")

if llm_available:
    df = df.merge(df_llm_sent[["meeting_date"] + llm_concept_cols], on="meeting_date", how="inner")
    print(f"After adding LLM sentiment: {len(df)} meetings")

# Restrict to paper's sample: 1982-10 to 2008-12
df = df[(df["meeting_date"] >= "1982-10-01") & (df["meeting_date"] <= "2008-12-31")]
print(f"After date filter (1982-10 to 2008-12): {len(df)} meetings")
print(f"\nFinal sample: {len(df)} meetings")
print(f"  FFR_change range: [{df['FFR_change'].min():.2f}, {df['FFR_change'].max():.2f}]")

## 1. Compute Sentiment PC1s

The paper uses the first principal component of all 296 sentiment indicators as a summary measure. We compute PC1 for both the dictionary-based and LLM-based sentiment matrices using identical methodology.

In [ ]:
def compute_pc1(df_in, cols, label=""):
    """Compute PC1 from sentiment columns. Returns PC1 array and explained variance ratio."""
    X = df_in[cols].values.astype(float)
    X = np.nan_to_num(X, nan=0.0)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    pca = PCA(n_components=1)
    pc1 = pca.fit_transform(X_scaled).flatten()
    evr = pca.explained_variance_ratio_[0]
    print(f"{label}: PC1 explains {evr:.1%} of variance, sign={'+' if pc1.mean() >= 0 else '-'}")
    return pc1, evr, pca.components_[0]

# Dictionary PC1
df["pc1_dict"], dict_evr, dict_loadings = compute_pc1(df, dict_concept_cols, "Dict PC1")

# LLM PC1 (if available)
if llm_available:
    df["pc1_llm"], llm_evr, llm_loadings = compute_pc1(df, llm_concept_cols, "LLM PC1")
    # Correlation between the two PC1s
    pc1_corr = np.corrcoef(df["pc1_dict"], df["pc1_llm"])[0, 1]
    print(f"\nCorrelation(dict_PC1, llm_PC1) = {pc1_corr:.4f}")
else:
    print("\nLLM data not available — skipping LLM PC1 computation.")

## 2. First-Stage Regression (Table 3 Framework)

Following the paper's methodology, we run Ridge regression of FFR target changes on Greenbook numerical forecasts + sentiment PC1. The deviance ratio (generalized R²) measures the share of FFR variation attributed to systematic monetary policy.

**Important**: We use a simplified specification (17 forecasts + 1 PC1) rather than the paper's full specification (3,226 variables with lags and nonlinear terms). This isolates the contribution of the sentiment PC1 and makes the comparison between dictionary and LLM scoring transparent.

In [ ]:
def run_ridge_regression(X, y, alphas=None):
    """Run Ridge regression with CV and return results dict."""
    if alphas is None:
        alphas = np.logspace(-2, 4, 30)
    
    mask = ~(np.isnan(X).any(axis=1) | np.isnan(y))
    X_clean = X[mask]
    y_clean = y[mask]
    
    if len(y_clean) < 20:
        return {"r2": None, "n": len(y_clean), "coef_pc1": None}
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_clean)
    
    ridge = RidgeCV(alphas=alphas, cv=5)
    ridge.fit(X_scaled, y_clean)
    
    r2 = ridge.score(X_scaled, y_clean)
    # Unscale coefficients to get coefficient on PC1
    coefs = ridge.coef_ / scaler.scale_
    # PC1 is the last column
    coef_pc1 = coefs[-1]
    
    return {
        "r2": round(float(r2), 4),
        "n": len(y_clean),
        "coef_pc1": round(float(coef_pc1), 4),
        "alpha": round(float(ridge.alpha_), 4),
        "coefs": coefs
    }

# Build feature matrices
X_forecasts = df[forecast_cols].values.astype(float)
X_forecasts = np.nan_to_num(X_forecasts, nan=0.0)
y = df["FFR_change"].values

# Dictionary specification
X_dict = np.column_stack([X_forecasts, df["pc1_dict"].values])
result_dict = run_ridge_regression(X_dict, y)
print(f"Dict: R²={result_dict['r2']:.4f}, PC1 coef={result_dict['coef_pc1']:.4f}, n={result_dict['n']}")

# LLM specification
if llm_available:
    X_llm = np.column_stack([X_forecasts, df["pc1_llm"].values])
    result_llm = run_ridge_regression(X_llm, y)
    print(f"LLM:  R²={result_llm['r2']:.4f}, PC1 coef={result_llm['coef_pc1']:.4f}, n={result_llm['n']}")
    print(f"ΔR² (LLM - Dict) = {result_llm['r2'] - result_dict['r2']:.4f}")
else:
    result_llm = {"r2": None}  # placeholder

---
## Check A: Full-Context LLM vs ±10-Word Dictionary Scoring

**Question**: Does the dictionary's ±10-word window lose information relative to full-document context?

**Design**: We hold everything constant EXCEPT the scoring method:
- **Same 296 concepts** (the exact set used in the dictionary sentiment matrix)
- **Same Ridge regression pipeline** (forecasts + PC1 → FFR change)
- **Same Greenbook document** (greenbook1, the staff qualitative assessment)
- **Different context**: dictionary uses ±10 words around each concept mention; LLM reads the full 8K–24K word document

**Interpretation**:
- If the two PC1s are highly correlated and produce similar R² → the ±10-word window is NOT a binding constraint; the dictionary method adequately captures sentiment
- If LLM PC1 produces meaningfully higher R² → the narrow window may miss relevant context; full-document reading adds information
- If LLM PC1 produces lower R² → full-context LLM scoring may introduce noise; the narrow window may help by focusing the signal

In [ ]:
check_a_rows = [
    {"Specification": "Forecasts only (no sentiment)",
     "R²": round(float(run_ridge_regression(X_forecasts, y)["r2"]), 4),
     "PC1 Source": "—"},
    {"Specification": "Forecasts + Dict PC1",
     "R²": result_dict["r2"],
     "PC1 Source": "LM Dictionary (±10 word window)"},
]
if llm_available:
    check_a_rows.append({
        "Specification": "Forecasts + LLM PC1",
        "R²": result_llm["r2"],
        "PC1 Source": "DeepSeek V4 Pro (full text)"
    })

df_check_a = pd.DataFrame(check_a_rows)
print("Table A: First-Stage Regression Fit Comparison")
print("=" * 65)
print(df_check_a.to_string(index=False))

if llm_available:
    # Visual comparison
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    
    # Bar chart
    ax = axes[0]
    specs = ["Forecasts\nonly", "Forecasts +\nDict PC1", "Forecasts +\nLLM PC1"]
    r2s = [check_a_rows[0]["R²"], check_a_rows[1]["R²"], check_a_rows[2]["R²"]]
    colors = ["#999999", "#4472C4", "#ED7D31"]
    bars = ax.bar(specs, r2s, color=colors, edgecolor="white", linewidth=1.2)
    for bar, val in zip(bars, r2s):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{val:.4f}", ha="center", va="bottom", fontsize=13, fontweight="bold")
    ax.set_ylabel("Deviance Ratio (R²)", fontsize=11)
    ax.set_title("First-Stage Regression Fit", fontsize=13, fontweight="bold")
    ax.set_ylim(0, max(r2s) * 1.2)
    ax.grid(axis="y", alpha=0.3)
    
    # Scatter: dict PC1 vs LLM PC1
    ax = axes[1]
    ax.scatter(df["pc1_dict"], df["pc1_llm"], alpha=0.6, s=30, color="#4472C4")
    ax.set_xlabel("Dictionary PC1", fontsize=11)
    ax.set_ylabel("LLM PC1", fontsize=11)
    ax.set_title(f"PC1 Comparison (r = {pc1_corr:.3f})", fontsize=13, fontweight="bold")
    ax.axhline(y=0, color="gray", linestyle="--", alpha=0.4)
    ax.axvline(x=0, color="gray", linestyle="--", alpha=0.4)
    ax.grid(alpha=0.3)
    
    fig.tight_layout()
    fig_path = PROJECT_ROOT / "results" / "figures" / "check_a_comparison.png"
    fig.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"\nFigure saved: {fig_path}")

---
## Check B: Sample Stability by Fed Chair

**Question**: Is the first-stage fit stable across different monetary policy regimes, or is it driven by a particular period?

We split the sample into three chairmanship periods:

| Period | Chair | Dates | Key Context |
|--------|-------|-------|-------------|
| 1982–1987 | Paul Volcker | High inflation / disinflation | Aggressive rate moves |
| 1987–2006 | Alan Greenspan | Great Moderation | Gradual, transparent policy |
| 2006–2008 | Ben Bernanke | Pre-crisis / early GFC | Low rates, emerging stress |

In [ ]:
periods = {
    "Volcker\n(1982–1987)": ("1982-10-01", "1987-08-15"),
    "Greenspan\n(1987–2006)": ("1987-08-16", "2006-01-31"),
    "Bernanke\n(2006–2008)": ("2006-02-01", "2008-12-31"),
}

check_b_rows = []
for period_label, (start, end) in periods.items():
    mask = (df["meeting_date"] >= start) & (df["meeting_date"] <= end)
    df_sub = df[mask]
    n = len(df_sub)
    
    if n < 15:
        print(f"{period_label}: n={n} — insufficient, skipping")
        continue
    
    y_sub = df_sub["FFR_change"].values
    X_fc_sub = df_sub[forecast_cols].values.astype(float)
    X_fc_sub = np.nan_to_num(X_fc_sub, nan=0.0)
    
    # Forecasts only
    r_fc = run_ridge_regression(X_fc_sub, y_sub)
    
    # Dict
    X_d_sub = np.column_stack([X_fc_sub, df_sub["pc1_dict"].values])
    r_d = run_ridge_regression(X_d_sub, y_sub)
    
    # LLM
    if llm_available:
        X_l_sub = np.column_stack([X_fc_sub, df_sub["pc1_llm"].values])
        r_l = run_ridge_regression(X_l_sub, y_sub)
    else:
        r_l = {"r2": None}
    
    check_b_rows.append({
        "Period": period_label.replace("\n", " "),
        "N": n,
        "Forecasts only R²": r_fc["r2"],
        "+ Dict PC1 R²": r_d["r2"],
        "+ LLM PC1 R²": r_l["r2"],
        "Δ (Dict - LLM)": round(r_d["r2"] - r_l["r2"], 4) if r_l["r2"] is not None else None,
    })
    
    print(f"{period_label.replace(chr(10),' ')}: n={n}, Fc={r_fc['r2']:.4f}, "
          f"+Dict={r_d['r2']:.4f}, +LLM={r_l['r2']}")

df_check_b = pd.DataFrame(check_b_rows)
print("\nTable B: Sample Stability Across Fed Chair Regimes")
print("=" * 75)
print(df_check_b.to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(check_b_rows))
width = 0.25

fc_r2s = [r["Forecasts only R²"] for r in check_b_rows]
dict_r2s = [r["+ Dict PC1 R²"] for r in check_b_rows]
llm_r2s = [r["+ LLM PC1 R²"] for r in check_b_rows]

bars1 = ax.bar(x - width, fc_r2s, width, label="Forecasts only", color="#999999", edgecolor="white")
bars2 = ax.bar(x, dict_r2s, width, label="+ Dict PC1", color="#4472C4", edgecolor="white")
if llm_available:
    bars3 = ax.bar(x + width, llm_r2s, width, label="+ LLM PC1", color="#ED7D31", edgecolor="white")

ax.set_ylabel("Deviance Ratio (R²)", fontsize=11)
ax.set_title("Fit by Fed Chair Regime", fontsize=13, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels([r["Period"] for r in check_b_rows], fontsize=10)
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig_path = PROJECT_ROOT / "results" / "figures" / "check_b_subsample.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"\nFigure saved: {fig_path}")

---
## Check C: Placebo Shuffle Test

**Question**: Is the predictive power of sentiment PC1 driven by the **identity** of specific economic concepts, or does it merely reflect the **overall tone** of the Greenbook text?

**Method**: Within each meeting, randomly permute the sentiment scores across concepts (breaking the concept–score correspondence while preserving the distribution). Recompute PC1, rerun the regression. Repeat 100 times.

**Interpretation**:
- If the actual R² is far outside the placebo distribution → the signal depends on **which** concepts are scored positively/negatively
- If the actual R² falls within the placebo distribution → the signal is mostly capturing general text tone

In [ ]:
N_PLACEBO = 100
np.random.seed(42)

dict_placebo_r2s = []
X_dict_sent_only = df[dict_concept_cols].values.astype(float)
X_dict_sent_only = np.nan_to_num(X_dict_sent_only, nan=0.0)

print(f"Running {N_PLACEBO} placebo shuffles for dictionary PC1...")
for seed in range(N_PLACEBO):
    rng = np.random.RandomState(seed)
    X_shuffled = np.zeros_like(X_dict_sent_only)
    for i in range(X_dict_sent_only.shape[0]):
        X_shuffled[i, :] = rng.permutation(X_dict_sent_only[i, :])
    
    # Compute PC1 from shuffled sentiments
    scaler = StandardScaler()
    X_s = scaler.fit_transform(X_shuffled)
    pc1_shuffled = PCA(n_components=1).fit_transform(X_s).flatten()
    
    X_p = np.column_stack([X_forecasts, pc1_shuffled])
    r_p = run_ridge_regression(X_p, y)
    dict_placebo_r2s.append(r_p["r2"])
    
    if (seed + 1) % 20 == 0:
        print(f"  {seed+1}/{N_PLACEBO} done")

dict_placebo_r2s = np.array(dict_placebo_r2s)
actual_r2 = result_dict["r2"]
p_value = (np.sum(dict_placebo_r2s >= actual_r2) + 1) / (N_PLACEBO + 1)

print(f"\nActual R² = {actual_r2:.4f}")
print(f"Placebo R²: mean={dict_placebo_r2s.mean():.4f}, "
      f"std={dict_placebo_r2s.std():.4f}, "
      f"max={dict_placebo_r2s.max():.4f}")
print(f"Empirical p-value = {p_value:.3f} ({'***' if p_value < 0.01 else '**' if p_value < 0.05 else '*' if p_value < 0.1 else 'n.s.'})")

# Histogram
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(dict_placebo_r2s, bins=25, color="#4472C4", alpha=0.7, edgecolor="white", label="Placebo R²")
ax.axvline(actual_r2, color="#C00000", linewidth=2.5, linestyle="--", label=f"Actual R² = {actual_r2:.4f}")
ax.set_xlabel("Deviance Ratio (R²)", fontsize=11)
ax.set_ylabel("Frequency", fontsize=11)
ax.set_title(f"Placebo Shuffle Test (N={N_PLACEBO})\np = {p_value:.3f}", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig_path = PROJECT_ROOT / "results" / "figures" / "check_c_placebo.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"\nFigure saved: {fig_path}")

---
## Check D: Shock Series Comparison

**Question**: If we compute monetary policy shocks (regression residuals) using dictionary-based vs LLM-based sentiment, how similar are the resulting shock series?

**Why this matters**: The shock series is the final output of the identification procedure — it's what gets used in impulse response functions and other macroeconomic analyses. If the two methods produce highly correlated shocks, researchers can be confident in the dictionary approach.

In [ ]:
def extract_shocks(X, y):
    """Extract regression residuals (= monetary policy shocks) from Ridge regression."""
    mask = ~(np.isnan(X).any(axis=1) | np.isnan(y))
    X_clean = X[mask]
    y_clean = y[mask]
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_clean)
    ridge = RidgeCV(alphas=np.logspace(-2, 4, 30), cv=5)
    ridge.fit(X_scaled, y_clean)
    y_pred = ridge.predict(X_scaled)
    residuals = y_clean - y_pred
    return residuals, mask

shocks_dict, mask_dict = extract_shocks(X_dict, y)
dates_valid = df["meeting_date"].values[mask_dict]
ffr_valid = y[mask_dict]

if llm_available:
    shocks_llm, mask_llm = extract_shocks(X_llm, y)
    # Use same mask for comparability
    common_mask = mask_dict & mask_llm
    shocks_dict_c = shocks_dict[np.where(common_mask[mask_dict])[0]] if not np.array_equal(mask_dict, mask_llm) else shocks_dict
    shocks_llm_c = shocks_llm[np.where(common_mask[mask_llm])[0]] if not np.array_equal(mask_dict, mask_llm) else shocks_llm
    
    shock_corr = np.corrcoef(shocks_dict_c, shocks_llm_c)[0, 1]
    print(f"Shock correlation: r = {shock_corr:.4f}")
    print(f"Dict shocks: mean={shocks_dict.mean():.4f}, std={shocks_dict.std():.4f}")
    print(f"LLM  shocks: mean={shocks_llm.mean():.4f}, std={shocks_llm.std():.4f}")
    print(f"Corr(dict_shock, ffr_change) = {np.corrcoef(shocks_dict, ffr_valid)[0,1]:.4f}")
    print(f"Corr(llm_shock,  ffr_change) = {np.corrcoef(shocks_llm, ffr_valid)[0,1]:.4f}")

# Plot shock series comparison
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

dates_dt = [datetime.strptime(d, "%Y-%m-%d") for d in dates_valid]

# Top: both shock series
ax = axes[0]
ax.plot(dates_dt, shocks_dict, "b-", linewidth=0.9, alpha=0.8, label="Dictionary-based shocks")
if llm_available:
    ax.plot(dates_dt, shocks_llm, "orange", linewidth=0.9, alpha=0.8, label="LLM-based shocks")
ax.axhline(y=0, color="gray", linestyle="-", alpha=0.4)
ax.set_ylabel("Shock (residual)", fontsize=11)
if llm_available:
    ax.set_title(f"Monetary Policy Shocks — Dict vs LLM (r = {shock_corr:.3f})", fontsize=13, fontweight="bold")
else:
    ax.set_title("Monetary Policy Shocks — Dictionary", fontsize=13, fontweight="bold")
ax.legend(fontsize=9, loc="upper right")
ax.grid(alpha=0.3)

# Bottom: scatter
ax = axes[1]
if llm_available:
    ax.scatter(shocks_dict_c, shocks_llm_c, alpha=0.5, s=25, color="#4472C4")
    # 45-degree line
    lim = max(abs(shocks_dict_c).max(), abs(shocks_llm_c).max()) * 1.1
    ax.plot([-lim, lim], [-lim, lim], "k--", alpha=0.3, linewidth=1)
    ax.set_xlabel("Dictionary Shock", fontsize=11)
    ax.set_ylabel("LLM Shock", fontsize=11)
    ax.set_title(f"Shock Correspondence (r = {shock_corr:.3f})", fontsize=13, fontweight="bold")
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.grid(alpha=0.3)

fig.tight_layout()
fig_path = PROJECT_ROOT / "results" / "figures" / "check_d_shocks.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"\nFigure saved: {fig_path}")

---
## Summary

### Key Findings

| Check | Key Result | Verdict |
|-------|-----------|---------|
| **A** — LLM vs Dictionary | | |
| **B** — Sample Stability | | |
| **C** — Placebo Shuffle | | |
| **D** — Shock Correlation | | |

### Bottom Line

(To be filled after running the full notebook with LLM scoring data.)

---

**Data sources**:
- FFR Target: `FFR_Target.csv` (267 FOMC meetings, 1982–2015)
- Greenbook forecasts: `Greenbook_Regressioninput_Original.csv` (17 numerical forecast variables)
- Dictionary sentiment: `total_sentiment_file_std10.csv` (296 concepts, ±10 word window, Loughran-McDonald dictionary)
- LLM sentiment: `sentiment_llm_v2.csv` (296 concepts scored by DeepSeek V4 Pro reading full greenbook1 text)
- Paper reference: Aruoba & Drechsel (2024), Table 3, *Econometrica*

### Design Note

This robustness check tests whether the dictionary's ±10-word window is a binding constraint on information extraction. The LLM reads the **entire greenbook1 document** (8K–24K words of prose — orders of magnitude more context than 20 words). The same 296 concepts are scored. Everything else (Ridge pipeline, forecast variables, sample period) is held constant. See `CLAUDE.md` for the full design rationale.